# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import numpy as np
import pandas as pd
from itertools import product
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import yaml
import duckdb
from datetime import datetime, timedelta
import time

import sys
sys.path.append(str(Path(globals()['_dh'][0]).resolve().parent.parent))

from generator.paths import project_path, pipeline_path, input_path, data_path, output_path
from generator.library.utilities import sort_params, clear_dir, ktok
from generator.library.loaders import base_loader_csv
from generator.library.progress import (
    load_progress, load_all_progress, save_progress, cleanup_progress, 
    get_base_scenario_key, is_base_scenario_complete,
    load_progress_with_timing, save_progress_with_timing, calculate_eta, get_timing_stats
)
from generator.transformers.dimension import segment_constant_factor_split, segment_add_total, segment_append_constant, segment_append_percentage, segment_append_reminder, segment_remove
from generator.transformers.scenario import create_base_scenario
from generator.library.db import read_data, delete_duckdb_file
import generator.library.scenario_constraints
from generator.library.scenario import process_scenario, process_scenario_sql

In [2]:
# 2. Load configuration

with open(project_path / 'config.yaml', "r") as f:
    config = yaml.safe_load(f)

with open(pipeline_path / 'county-pipeline.yaml', "r") as f:
    pipeline = yaml.safe_load(f)

## Set flags to clear directories
clear_output_flag = pipeline['clear_ouptput_flag']
clear_api_flag = pipeline['clear_api_flag']

## CONDITIONAL OUTPUT PATH: Write to /api/data if useAPI=true, else /generator/output
if config.get('useAPI', False):
    # Write directly to API data directory (eliminates 13-minute copy)
    parquet_output_path = data_path
    print(f"🚀 useAPI=true: Writing parquet data directly to API directory: {data_path}")
else:
    # Write to generator output directory
    parquet_output_path = output_path
    print(f"📁 useAPI=false: Writing parquet data to generator directory: {output_path}")

## Set common variables from pipeline config
db_file = output_path / pipeline['database']['file']  # Always keep temp DB in output_path
default_table = pipeline['database']['default_table']

full_partition = pipeline['partition']['full_partition']

base_schema = list(pipeline['base_data']['schema'].keys())
base_schema_map = pipeline['base_data']['schema']

base_scenario = pipeline['base_scenario']['name']
include_base_scenario = pipeline['base_scenario']['include_base_scenario']

# Extract scenario schema from config.yaml
scenario_schema = [scenario['name'] for scenario in config['scenario']['scenarios']]

# Use scenario_params from pipeline config
scenario_params = {}
for name, params in pipeline['scenario_params'].items():
    scenario_params[name] = {
        **params,
        'curve_path_template': str(input_path / params['curve_path_template'])
    }

full_schema = scenario_schema + base_schema

base_data = pipeline['base_data']['file']

print(f"📁 Temporary database: {db_file}")
print(f"📊 Parquet output: {parquet_output_path}")
print(f"⚙️  Progress tracking: {parquet_output_path}")

🚀 useAPI=true: Writing parquet data directly to API directory: /home/viktor/code/behovskartan/api/data
📁 Temporary database: /home/viktor/code/behovskartan/generator/output/core.duckdb
📊 Parquet output: /home/viktor/code/behovskartan/api/data
⚙️  Progress tracking: /home/viktor/code/behovskartan/api/data


In [3]:
# Clear directories BEFORE generation (for fresh start)

# Clear the generator output directory if requested
if clear_output_flag:
    clear_dir(output_path)

# Clear API data directory BEFORE generation (preserving static files)
if clear_api_flag and config.get('useAPI', False):
    import shutil
    
    print("🧹 Clearing API data directory before generation (preserving static files)...")
    
    # Files to preserve (static API files that shouldn't be deleted)
    static_files = [
        'parameters.json',
        'config.json', 
        'scenarios.json',
        'aggregations.json',
        'geographies.json',
        'geographies.geojson'
    ]
    
    preserved_files = {}
    
    # Backup static files if they exist
    for filename in static_files:
        filepath = data_path / filename
        if filepath.exists():
            with open(filepath, 'rb') as f:
                preserved_files[filename] = f.read()
            print(f"  📄 Preserved: {filename}")
    
    # Clear the directory
    if data_path.exists():
        clear_dir(data_path)
    else:
        data_path.mkdir(parents=True, exist_ok=True)
    
    # Restore preserved files
    for filename, content in preserved_files.items():
        filepath = data_path / filename
        with open(filepath, 'wb') as f:
            f.write(content)
        print(f"  ✅ Restored: {filename}")
    
    if preserved_files:
        print(f"🔒 Preserved {len(preserved_files)} static API files during cleanup")
    else:
        print("⚠️  No static API files found to preserve (you may need to run setup script)")
        
elif clear_api_flag and not config.get('useAPI', False):
    print("🧹 Clearing generator output directory before generation...")
    clear_dir(output_path)

In [4]:
# 3. Calculate the scenarios

# Extract names and value lists
names = list(scenario_params.keys())
values = [scenario_params[name]['parameters'] for name in names]

# Define default scenario
default_scenario = {
    name: scenario_params[name]["default"]
    for name in names
}

# Generate all combinations
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Mark the default is base is not kept (and is default)
if include_base_scenario:
    scenarios = all_scenarios
else:
    scenarios = []
    for s in all_scenarios:
        s_out = dict(s)
        if s == default_scenario:
            s_out["default"] = True
        scenarios.append(s_out)

In [5]:
# 3. Load the base demand

base_loader_csv(base_schema, base_schema_map, input_path / base_data, db_file)

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'demand',
 'added_rows': 8760,
 'added_columns': ['geography', 'segment', 'timestamp', 'value']}

In [6]:
# 4 Segment the data (using pipeline config)

# 4.1 Split the data by geographies
geo_file = input_path / pipeline['geography_split']['file']
geographies = pd.read_csv(geo_file, dtype={"geography": str, "factor": float})

segment_constant_factor_split(db_file, geographies)

# 4.2 Split the data by segments (using pipeline operations)
for operation in pipeline['segment_processing']['operations']:
    if operation['type'] == 'append_constant':
        segment_append_constant(db_file, 'segment', operation['segment'], operation['value'])
    elif operation['type'] == 'append_percentage':
        segment_append_percentage(db_file, 'segment', operation['from_segment'], operation['to_segment'], operation['percentage'])
    elif operation['type'] == 'append_reminder':
        segment_append_reminder(db_file, 'segment', operation['from_segment'], operation['exclude_segments'], operation['to_segment'])
    elif operation['type'] == 'remove':
        segment_remove(db_file, 'segment', operation['segment'])

In [8]:
# Create base scenario (using conditional output path and structure)

if USE_NESTED_STRUCTURE:
    from generator.transformers.scenario import create_base_scenario_nested
    
    print(f"🔨 Creating base scenario with NEW base structure...")
    print(f"   📁 Structure: base/{base_scenario}/data.parquet")
    
    # Check if base scenario already exists in base structure
    expected_base_path = parquet_output_path / "base" / base_scenario
    expected_base_file = expected_base_path / "data.parquet"
    
    if expected_base_file.exists():
        print(f"✅ Base scenario '{base_scenario}' already exists in base structure, skipping creation")
        print(f"   📁 Location: {expected_base_file}")
    else:
        result = create_base_scenario_nested(
            in_data=db_file, 
            base_year=pipeline['base_scenario']['base_year'],
            start_year=pipeline['base_scenario']['start_year'],
            end_year=pipeline['base_scenario']['end_year'],
            out_data=parquet_output_path,  # Use conditional output path
            growth_curve=input_path / pipeline['growth_curve']['file'],
            base_scenario=base_scenario
        )
        
        print(f"✅ Base scenario completed with base structure")
        print(result)
        
else:
    from generator.transformers.scenario import create_base_scenario
    
    # Check if base scenario already exists using library function
    if is_base_scenario_complete(parquet_output_path, base_scenario):
        print(f"✅ Base scenario '{base_scenario}' already exists, skipping creation")
    else:
        print(f"🔨 Creating base scenario '{base_scenario}'...")
        result = create_base_scenario(
            in_data=db_file, 
            base_year=pipeline['base_scenario']['base_year'],
            start_year=pipeline['base_scenario']['start_year'],
            end_year=pipeline['base_scenario']['end_year'],
            out_data=parquet_output_path,  # Use conditional output path
            partition=full_partition,
            growth_curve=input_path / pipeline['growth_curve']['file'],
            base_scenario=base_scenario,
            scenario_schema=scenario_schema
        )
        
        # Mark base scenario as completed using library function
        base_scenario_key = get_base_scenario_key(base_scenario)
        save_progress(parquet_output_path, base_scenario_key)
        
        print(f"✅ Base scenario completed and saved to progress")
        print(result)

🔨 Creating base scenario with NEW base structure...
   📁 Structure: base/default/data.parquet
  ✅ Base scenario written to: /home/viktor/code/behovskartan/api/data/base/default/data.parquet
✅ Base scenario completed with base structure
{'target': '/home/viktor/code/behovskartan/api/data/base/default/data.parquet', 'status': 'done', 'rows_written': 14357952, 'years': {'start': 2025, 'end': 2050}, 'scenario_id': 'default', 'structure': 'nested'}


In [ ]:
# Create base scenario (using conditional output path)

from generator.transformers.scenario import create_base_scenario_nested

print(f"🔨 Creating base scenario with nested structure...")
print(f"   📁 Structure: base/{base_scenario}/data.parquet")

# Check if base scenario already exists in base structure
expected_base_path = parquet_output_path / "base" / base_scenario
expected_base_file = expected_base_path / "data.parquet"

if expected_base_file.exists():
    print(f"✅ Base scenario '{base_scenario}' already exists, skipping creation")
    print(f"   📁 Location: {expected_base_file}")
else:
    result = create_base_scenario_nested(
        in_data=db_file, 
        base_year=pipeline['base_scenario']['base_year'],
        start_year=pipeline['base_scenario']['start_year'],
        end_year=pipeline['base_scenario']['end_year'],
        out_data=parquet_output_path,  # Use conditional output path
        growth_curve=input_path / pipeline['growth_curve']['file'],
        base_scenario=base_scenario
    )
    
    print(f"✅ Base scenario completed")
    print(result)

In [10]:
# Resumable scenario processing (using conditional output path)

if USE_NESTED_STRUCTURE:
    from generator.library.scenario import process_scenario_nested_sql
    print("🚀 Using NEW nested structure (one file per scenario)")
    print("   Structure: scenarios/param1=value1/param2=value2/.../data.parquet")
    print("   Expected 20-100x performance improvement")
else:
    from generator.library.scenario import process_scenario_sql
    print("📁 Using legacy partitioned structure")

# Resume-aware scenario processing using library functions with timing
scenario_ids = []
progress_data = load_progress_with_timing(parquet_output_path)
completed_scenarios = progress_data['completed']
total_scenarios = len(scenarios)
initial_completed_count = len(completed_scenarios)

print(f"Found {initial_completed_count} already completed scenarios out of {total_scenarios} total")

# Show timing statistics if available
if progress_data['timing_data']:
    stats = get_timing_stats(parquet_output_path)
    print(f"📊 Timing stats: avg {stats['average']:.1f}s, min {stats['min']:.1f}s, max {stats['max']:.1f}s (from {stats['count']} scenarios)")

# Filter out completed scenarios
remaining_scenarios = []
for scn in scenarios:
    scenario_id = ",".join(f"{k}={scn[k]}" for k in scenario_schema)
    if scenario_id not in completed_scenarios:
        remaining_scenarios.append((scn, scenario_id))
    else:
        scenario_ids.append(scenario_id)  # Add to completed list

print(f"Remaining scenarios to process: {len(remaining_scenarios)}")

# Process remaining scenarios
for i, (scn, scenario_id) in enumerate(remaining_scenarios):
    current_position = initial_completed_count + i + 1
    print(f"[{current_position}/{total_scenarios}] Processing scenario: {scenario_id}")
    start_time = time.perf_counter()
    
    try:
        if USE_NESTED_STRUCTURE:
            # Use new nested structure function
            sid = process_scenario_nested_sql(
                scenario=scn,
                scenario_params=scenario_params,
                scenario_schema=scenario_schema,
                output_path=parquet_output_path,  # Use conditional output path
                base_scenario=base_scenario,
            )
        else:
            # Use legacy partitioned function
            sid = process_scenario_sql(
                scenario=scn,
                scenario_params=scenario_params,
                scenario_schema=scenario_schema,
                output_path=parquet_output_path,  # Use conditional output path
                partition=full_partition,
                base_scenario=base_scenario,
            )
        
        # Calculate elapsed time and save with timing data
        elapsed = time.perf_counter() - start_time
        scenario_ids.append(sid)
        completed_scenarios.add(scenario_id)
        save_progress_with_timing(parquet_output_path, scenario_id, elapsed)
        
        print(f"  ✅ Finished in {elapsed:.2f}s")
        
        # Improved ETA calculation based on historical data
        remaining = len(remaining_scenarios) - (i + 1)
        if remaining > 0:
            eta_seconds, sample_size = calculate_eta(parquet_output_path, remaining)
            eta_minutes = eta_seconds / 60
            current_total_completed = initial_completed_count + i + 1
            
            if sample_size > 0:
                print(f"  📊 Progress: {current_total_completed}/{total_scenarios} | ETA: {eta_minutes:.1f} minutes (based on {sample_size} samples)")
            else:
                # Fallback to current scenario timing if no historical data
                fallback_eta = elapsed * remaining / 60
                print(f"  📊 Progress: {current_total_completed}/{total_scenarios} | ETA: {fallback_eta:.1f} minutes (estimate)")
            
    except KeyboardInterrupt:
        current_total_completed = initial_completed_count + i + 1
        print(f"\n⚠️  Interrupted! Processed {current_total_completed}/{total_scenarios} scenarios")
        print(f"Progress saved. To resume, simply re-run this cell.")
        break
    except Exception as e:
        print(f"  ❌ Error processing scenario {scenario_id}: {e}")
        continue

# Clean up progress file if all scenarios completed using library function
final_completed = len(completed_scenarios)
if final_completed == total_scenarios:
    if cleanup_progress(parquet_output_path):
        print("🧹 Cleaned up progress tracking file")

print(f"\n🎉 Final status: {final_completed}/{total_scenarios} scenarios completed")

# Show final timing summary
if final_completed > 0:
    final_stats = get_timing_stats(parquet_output_path)
    total_time_hours = final_stats['total'] / 3600
    print(f"⏱️  Total processing time: {total_time_hours:.2f} hours | Average: {final_stats['average']:.1f}s per scenario")

print("Scenarios written (SQL):", scenario_ids)

🚀 Using NEW nested structure (one file per scenario)
   Structure: scenarios/param1=value1/param2=value2/.../data.parquet
   Expected 20-100x performance improvement
Found 0 already completed scenarios out of 75 total
Remaining scenarios to process: 75
[1/75] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=0
[SQL] Processing scenario (nested): housing_electrification=0,transport_electrification=0,industry_transition=0
  📁 Using base structure scenario: /home/viktor/code/behovskartan/api/data/base/default/data.parquet


  ✅ Written to: /home/viktor/code/behovskartan/api/data/scenarios/housing_electrification=0/transport_electrification=0/industry_transition=0/data.parquet
  ✅ Finished in 2.25s
  📊 Progress: 1/75 | ETA: 2.8 minutes (based on 1 samples)
[2/75] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=1
[SQL] Processing scenario (nested): housing_electrification=0,transport_electrification=0,industry_transition=1
  📁 Using base structure scenario: /home/viktor/code/behovskartan/api/data/base/default/data.parquet
  ✅ Written to: /home/viktor/code/behovskartan/api/data/scenarios/housing_electrification=0/transport_electrification=0/industry_transition=1/data.parquet
  ✅ Finished in 2.23s
  📊 Progress: 2/75 | ETA: 2.7 minutes (based on 2 samples)
[3/75] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=2
[SQL] Processing scenario (nested): housing_electrification=0,transport_electrification=0,industry_transiti

In [ ]:
# Resumable scenario processing (using conditional output path)

from generator.library.scenario import process_scenario_nested_sql

print("🚀 Using nested structure (one file per scenario)")
print("   Structure: scenarios/param1=value1/param2=value2/.../data.parquet")

# Resume-aware scenario processing using library functions with timing
scenario_ids = []
progress_data = load_progress_with_timing(parquet_output_path)
completed_scenarios = progress_data['completed']
total_scenarios = len(scenarios)
initial_completed_count = len(completed_scenarios)

print(f"Found {initial_completed_count} already completed scenarios out of {total_scenarios} total")

# Show timing statistics if available
if progress_data['timing_data']:
    stats = get_timing_stats(parquet_output_path)
    print(f"📊 Timing stats: avg {stats['average']:.1f}s, min {stats['min']:.1f}s, max {stats['max']:.1f}s (from {stats['count']} scenarios)")

# Filter out completed scenarios
remaining_scenarios = []
for scn in scenarios:
    scenario_id = ",".join(f"{k}={scn[k]}" for k in scenario_schema)
    if scenario_id not in completed_scenarios:
        remaining_scenarios.append((scn, scenario_id))
    else:
        scenario_ids.append(scenario_id)  # Add to completed list

print(f"Remaining scenarios to process: {len(remaining_scenarios)}")

# Process remaining scenarios
for i, (scn, scenario_id) in enumerate(remaining_scenarios):
    current_position = initial_completed_count + i + 1
    print(f"[{current_position}/{total_scenarios}] Processing scenario: {scenario_id}")
    start_time = time.perf_counter()
    
    try:
        # Use nested structure function
        sid = process_scenario_nested_sql(
            scenario=scn,
            scenario_params=scenario_params,
            scenario_schema=scenario_schema,
            output_path=parquet_output_path,  # Use conditional output path
            base_scenario=base_scenario,
        )
        
        # Calculate elapsed time and save with timing data
        elapsed = time.perf_counter() - start_time
        scenario_ids.append(sid)
        completed_scenarios.add(scenario_id)
        save_progress_with_timing(parquet_output_path, scenario_id, elapsed)
        
        print(f"  ✅ Finished in {elapsed:.2f}s")
        
        # Improved ETA calculation based on historical data
        remaining = len(remaining_scenarios) - (i + 1)
        if remaining > 0:
            eta_seconds, sample_size = calculate_eta(parquet_output_path, remaining)
            eta_minutes = eta_seconds / 60
            current_total_completed = initial_completed_count + i + 1
            
            if sample_size > 0:
                print(f"  📊 Progress: {current_total_completed}/{total_scenarios} | ETA: {eta_minutes:.1f} minutes (based on {sample_size} samples)")
            else:
                # Fallback to current scenario timing if no historical data
                fallback_eta = elapsed * remaining / 60
                print(f"  📊 Progress: {current_total_completed}/{total_scenarios} | ETA: {fallback_eta:.1f} minutes (estimate)")
            
    except KeyboardInterrupt:
        current_total_completed = initial_completed_count + i + 1
        print(f"\n⚠️  Interrupted! Processed {current_total_completed}/{total_scenarios} scenarios")
        print(f"Progress saved. To resume, simply re-run this cell.")
        break
    except Exception as e:
        print(f"  ❌ Error processing scenario {scenario_id}: {e}")
        continue

# Clean up progress file if all scenarios completed using library function
final_completed = len(completed_scenarios)
if final_completed == total_scenarios:
    if cleanup_progress(parquet_output_path):
        print("🧹 Cleaned up progress tracking file")

print(f"\n🎉 Final status: {final_completed}/{total_scenarios} scenarios completed")

# Show final timing summary
if final_completed > 0:
    final_stats = get_timing_stats(parquet_output_path)
    total_time_hours = final_stats['total'] / 3600
    print(f"⏱️  Total processing time: {total_time_hours:.2f} hours | Average: {final_stats['average']:.1f}s per scenario")

print("Scenarios written (SQL):", scenario_ids)

In [ ]:
# Generate aggregated tables for fast queries

if final_completed == total_scenarios:
    print("\n📊 Generating aggregated tables for fast queries...")
    
    import duckdb
    con = duckdb.connect()
    
    # Scan both base and scenario files
    base_glob = str(parquet_output_path / "base" / "**" / "data.parquet")
    scenarios_glob = str(parquet_output_path / "scenarios" / "**" / "data.parquet")
    aggregated_dir = parquet_output_path / "aggregated"
    aggregated_dir.mkdir(exist_ok=True)
    
    try:
        # Create normalized UNION query to handle schema differences
        # Base scenarios: add NULL parameter columns to match scenarios schema
        # Scenarios: use existing parameter columns
        union_query = f"""
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                NULL as housing_electrification,
                NULL as transport_electrification, 
                NULL as industry_transition,
                true as is_base
            FROM parquet_scan('{base_glob}', hive_partitioning=FALSE)
            UNION ALL
            SELECT 
                timestamp, value, geography, segment, scenario_id,
                housing_electrification,
                transport_electrification, 
                industry_transition,
                false as is_base
            FROM parquet_scan('{scenarios_glob}', hive_partitioning=FALSE)
        """
        
        # 1. Yearly totals by geography (for map visualization)
        print("  🗺️  Creating yearly geography totals...")
        con.execute(f"""
            COPY (
                SELECT 
                    COALESCE(CAST(housing_electrification AS VARCHAR), 'N/A') as housing_electrification,
                    COALESCE(CAST(transport_electrification AS VARCHAR), 'N/A') as transport_electrification, 
                    COALESCE(CAST(industry_transition AS VARCHAR), 'N/A') as industry_transition,
                    scenario_id,
                    geography,
                    strftime(timestamp, '%Y') as year,
                    SUM(value) as total_value,
                    is_base
                FROM ({union_query}) combined_data
                GROUP BY housing_electrification, transport_electrification, industry_transition, 
                         scenario_id, geography, strftime(timestamp, '%Y'), is_base
            ) TO '{aggregated_dir / "geography_yearly.parquet"}'
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 3)
        """)
        
        # 2. Yearly totals by segment (for sector charts)
        print("  📊 Creating yearly segment totals...")
        con.execute(f"""
            COPY (
                SELECT 
                    COALESCE(CAST(housing_electrification AS VARCHAR), 'N/A') as housing_electrification,
                    COALESCE(CAST(transport_electrification AS VARCHAR), 'N/A') as transport_electrification, 
                    COALESCE(CAST(industry_transition AS VARCHAR), 'N/A') as industry_transition,
                    scenario_id,
                    segment,
                    strftime(timestamp, '%Y') as year,
                    SUM(value) as total_value,
                    is_base
                FROM ({union_query}) combined_data
                GROUP BY housing_electrification, transport_electrification, industry_transition, 
                         scenario_id, segment, strftime(timestamp, '%Y'), is_base
            ) TO '{aggregated_dir / "segment_yearly.parquet"}'
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 3)
        """)
        
        # 3. National yearly totals (for time series)
        print("  🏛️  Creating national yearly totals...")
        con.execute(f"""
            COPY (
                SELECT 
                    COALESCE(CAST(housing_electrification AS VARCHAR), 'N/A') as housing_electrification,
                    COALESCE(CAST(transport_electrification AS VARCHAR), 'N/A') as transport_electrification, 
                    COALESCE(CAST(industry_transition AS VARCHAR), 'N/A') as industry_transition,
                    scenario_id,
                    strftime(timestamp, '%Y') as year,
                    SUM(value) as total_value,
                    is_base
                FROM ({union_query}) combined_data
                GROUP BY housing_electrification, transport_electrification, industry_transition, 
                         scenario_id, strftime(timestamp, '%Y'), is_base
            ) TO '{aggregated_dir / "national_yearly.parquet"}'
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 3)
        """)
        
        # 4. Scenario metadata (for quick scenario lookups)
        print("  🏷️  Creating scenario metadata...")
        con.execute(f"""
            COPY (
                SELECT DISTINCT
                    COALESCE(CAST(housing_electrification AS VARCHAR), 'N/A') as housing_electrification,
                    COALESCE(CAST(transport_electrification AS VARCHAR), 'N/A') as transport_electrification, 
                    COALESCE(CAST(industry_transition AS VARCHAR), 'N/A') as industry_transition,
                    scenario_id,
                    is_base
                FROM ({union_query}) combined_data
            ) TO '{aggregated_dir / "scenario_metadata.parquet"}'
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 3)
        """)
        
        print("✅ Aggregated tables created successfully!")
        print(f"   📁 Location: {aggregated_dir}")
        print("   🚀 These will provide 50-100x faster queries for common operations")
        print("   📊 Tables include both base scenarios and parametric scenarios")
        
    except Exception as e:
        print(f"❌ Error creating aggregated tables: {e}")
    finally:
        con.close()
        
else:
    print("ℹ️  Skipping aggregated table generation (not all scenarios completed)")